# 🚀 Quest 3S Multi-Dataset Teacher-Student Distillation on Dual Tesla T4
### 4 Canonical Datasets: Flickr1024 + Middlebury + KITTI 2012 + KITTI 2015
- **Hardware Target:** Meta Quest 3S (Snapdragon XR2 Gen 2 / Adreno 740 / 1024 KB GMEM SRAM)
- **Student Model:** Epipolar Windowed DyT-NAFRSSR v2 (82,368 params, 100% SRAM resident, 1.38 ms)
- **Teacher Model:** Official NAFSSR-M (280,383 params, 29.59 dB Middlebury, CVPR/ECCV)
- **Losses:** Charbonnier L1 + High-Frequency Sobel Edge Loss + Feature Distillation Loss
- **Augmentations:** Random Horizontal/Vertical Flips + Channel Shuffle (from NAFSSR ECCV 2022)

In [ ]:
!nvidia-smi
!pip install -q onnx onnxscript
import os, sys, time, math, random, glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np

num_gpus = torch.cuda.device_count()
print(f'[*] Allocated GPUs: {num_gpus}')
for i in range(num_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f'    [GPU {i}] {p.name} | VRAM: {p.total_memory / (1024**3):.2f} GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 2. Clone Official NAFRSSR Teacher Repository & Weights

In [ ]:
!git clone --depth 1 https://github.com/JNUChenYiHong/NAFRSSR.git /kaggle/working/NAFRSSR
sys.path.insert(0, '/kaggle/working/NAFRSSR')
try:
    from NAFRSSR_M_Net import NAFSSR as TeacherNAFSSR
    teacher_model = TeacherNAFSSR(up_scale=4, width=64, num_blks=16).to(device)
    teacher_ckpt = torch.load('/kaggle/working/NAFRSSR/NAFRSSR_M.pth', map_location=device)
    teacher_model.load_state_dict(teacher_ckpt['params'] if 'params' in teacher_ckpt else teacher_ckpt)
    teacher_model.eval()
    for p in teacher_model.parameters():
        p.requires_grad = False
    print('[*] Official Teacher NAFSSR-M (280K params) loaded and frozen!')
except Exception as e:
    print(f'[WARN] Teacher loading: {e}')
    teacher_model = None


## 3. Universal Multi-Dataset Loader (Flickr1024 + Middlebury + KITTI 2012/2015)

In [ ]:
class MultiCanonicalStereoDataset(Dataset):
    def __init__(self, crop_lr=48, upscale=4, is_train=True, max_pairs=1500):
        self.crop_lr = crop_lr
        self.crop_hr = crop_lr * upscale
        self.upscale = upscale
        self.is_train = is_train
        self.pairs = []

        # 1. Scan for Flickr1024
        for lf in glob.glob('/kaggle/input/**/*_L.png', recursive=True):
            rf = lf.replace('_L.png', '_R.png')
            if os.path.exists(rf):
                self.pairs.append((lf, rf))

        # 2. Scan for KITTI 2012 / 2015
        for lf in glob.glob('/kaggle/input/**/colored_0/*.png', recursive=True):
            rf = lf.replace('colored_0', 'colored_1')
            if os.path.exists(rf):
                self.pairs.append((lf, rf))
        for lf in glob.glob('/kaggle/input/**/image_2/*.png', recursive=True):
            rf = lf.replace('image_2', 'image_3')
            if os.path.exists(rf):
                self.pairs.append((lf, rf))

        # 3. Scan for Middlebury
        for lf in glob.glob('/kaggle/input/**/im0.png', recursive=True):
            rf = lf.replace('im0.png', 'im1.png')
            if os.path.exists(rf):
                self.pairs.append((lf, rf))
        for lf in glob.glob('/kaggle/input/**/view1.png', recursive=True):
            rf = lf.replace('view1.png', 'view5.png')
            if os.path.exists(rf):
                self.pairs.append((lf, rf))

        random.seed(42)
        random.shuffle(self.pairs)
        if is_train:
            self.pairs = self.pairs[:max_pairs]
        else:
            self.pairs = self.pairs[-50:]
        print(f'[*] Multi-Dataset loaded {len(self.pairs)} stereo pairs (is_train={is_train})')

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        lf, rf = self.pairs[idx]
        try:
            img_l = Image.open(lf).convert('RGB')
            img_r = Image.open(rf).convert('RGB')
            arr_l = np.array(img_l, dtype=np.float32) / 255.0
            arr_r = np.array(img_r, dtype=np.float32) / 255.0
            t_l = torch.from_numpy(arr_l).permute(2, 0, 1)
            t_r = torch.from_numpy(arr_r).permute(2, 0, 1)
        except Exception:
            t_l = torch.zeros(3, 192, 192)
            t_r = torch.zeros(3, 192, 192)

        H, W = t_l.shape[1], t_l.shape[2]
        if self.is_train and H >= self.crop_hr and W >= self.crop_hr:
            top = random.randint(0, H - self.crop_hr)
            left = random.randint(0, W - self.crop_hr)
            p_hr_l = t_l[:, top:top+self.crop_hr, left:left+self.crop_hr]
            p_hr_r = t_r[:, top:top+self.crop_hr, left:left+self.crop_hr]
        else:
            cy = max(0, (H - self.crop_hr) // 2)
            cx = max(0, (W - self.crop_hr) // 2)
            p_hr_l = t_l[:, cy:cy+self.crop_hr, cx:cx+self.crop_hr]
            p_hr_r = t_r[:, cy:cy+self.crop_hr, cx:cx+self.crop_hr]

        if p_hr_l.shape[1] != self.crop_hr or p_hr_l.shape[2] != self.crop_hr:
            p_hr_l = F.interpolate(p_hr_l.unsqueeze(0), size=(self.crop_hr, self.crop_hr), mode='bicubic', align_corners=False).squeeze(0)
            p_hr_r = F.interpolate(p_hr_r.unsqueeze(0), size=(self.crop_hr, self.crop_hr), mode='bicubic', align_corners=False).squeeze(0)

        if self.is_train:
            # Aug 1: Horizontal Flip
            if random.random() > 0.5:
                p_hr_l, p_hr_r = torch.flip(p_hr_r, dims=[2]), torch.flip(p_hr_l, dims=[2])
            # Aug 2: Vertical Flip
            if random.random() > 0.5:
                p_hr_l = torch.flip(p_hr_l, dims=[1])
                p_hr_r = torch.flip(p_hr_r, dims=[1])
            # Aug 3: Channel Shuffle (from NAFSSR ECCV 2022)
            if random.random() > 0.5:
                perm = torch.randperm(3)
                p_hr_l = p_hr_l[perm]
                p_hr_r = p_hr_r[perm]

        p_lr_l = F.interpolate(p_hr_l.unsqueeze(0), scale_factor=0.25, mode='bicubic', align_corners=False).squeeze(0).clamp(0, 1)
        p_lr_r = F.interpolate(p_hr_r.unsqueeze(0), scale_factor=0.25, mode='bicubic', align_corners=False).squeeze(0).clamp(0, 1)

        return {'lr_l': p_lr_l, 'lr_r': p_lr_r, 'hr_l': p_hr_l, 'hr_r': p_hr_r}


## 4. Student Model: Epipolar Windowed DyT-NAFRSSR v2 with Fixup Zero-Init

In [ ]:
class DynamicTanh(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.gamma = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, channels, 1, 1))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.gamma * torch.tanh(self.alpha * x) + self.beta

class SimpleGate(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class WindowedSCAM(nn.Module):
    def __init__(self, c: int, min_disp: int = -2, max_disp: int = 14):
        super().__init__()
        self.min_disp = min_disp
        self.max_disp = max_disp
        self.scale = c ** -0.5
        self.q_l = nn.Conv2d(c, c, kernel_size=1, bias=False)
        self.k_r = nn.Conv2d(c, c, kernel_size=1, bias=False)
        self.v_r = nn.Conv2d(c, c, kernel_size=1, bias=False)
        self.q_r = nn.Conv2d(c, c, kernel_size=1, bias=False)
        self.k_l = nn.Conv2d(c, c, kernel_size=1, bias=False)
        self.v_l = nn.Conv2d(c, c, kernel_size=1, bias=False)
        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))
    def forward(self, x_l: torch.Tensor, x_r: torch.Tensor):
        ql = self.q_l(x_l)
        kr = self.k_r(x_r)
        vr = self.v_r(x_r)
        qr = self.q_r(x_r)
        kl = self.k_l(x_l)
        vl = self.v_l(x_l)
        corr_l2r, corr_r2l, shifted_v_r, shifted_v_l = [], [], [], []
        for d in range(self.min_disp, self.max_disp + 1):
            if d >= 0:
                kr_s = F.pad(kr[:, :, :, d:], (0, d, 0, 0))
                vr_s = F.pad(vr[:, :, :, d:], (0, d, 0, 0))
            else:
                kr_s = F.pad(kr[:, :, :, :d], (-d, 0, 0, 0))
                vr_s = F.pad(vr[:, :, :, :d], (-d, 0, 0, 0))
            corr_l2r.append(torch.sum(ql * kr_s, dim=1, keepdim=True) * self.scale)
            shifted_v_r.append(vr_s)
            d_rev = -d
            if d_rev >= 0:
                kl_s = F.pad(kl[:, :, :, d_rev:], (0, d_rev, 0, 0))
                vl_s = F.pad(vl[:, :, :, d_rev:], (0, d_rev, 0, 0))
            else:
                kl_s = F.pad(kl[:, :, :, :d_rev], (-d_rev, 0, 0, 0))
                vl_s = F.pad(vl[:, :, :, :d_rev], (-d_rev, 0, 0, 0))
            corr_r2l.append(torch.sum(qr * kl_s, dim=1, keepdim=True) * self.scale)
            shifted_v_l.append(vl_s)
        w_l2r = F.softmax(torch.cat(corr_l2r, dim=1), dim=1)
        f_r2l = torch.sum(torch.stack(shifted_v_r, dim=1) * w_l2r.unsqueeze(2), dim=1)
        w_r2l = F.softmax(torch.cat(corr_r2l, dim=1), dim=1)
        f_l2r = torch.sum(torch.stack(shifted_v_l, dim=1) * w_r2l.unsqueeze(2), dim=1)
        return x_l + f_r2l * self.beta, x_r + f_l2r * self.gamma

class DyTNAFBlock(nn.Module):
    def __init__(self, c: int, dw_expand: int = 2, ffn_expand: int = 2):
        super().__init__()
        dw_c = c * dw_expand
        self.conv1 = nn.Conv2d(c, dw_c, kernel_size=1)
        self.conv2 = nn.Conv2d(dw_c, dw_c, kernel_size=3, padding=1, groups=dw_c)
        self.conv3 = nn.Conv2d(dw_c // 2, c, kernel_size=3, padding=1, groups=c // 4)
        self.sg = SimpleGate()
        ffn_c = ffn_expand * c
        self.conv4 = nn.Conv2d(c, ffn_c, kernel_size=1)
        self.conv5 = nn.Conv2d(ffn_c // 2, c, kernel_size=1)
        self.norm1 = DynamicTanh(c)
        self.norm2 = DynamicTanh(c)
        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))
    def forward(self, inp: torch.Tensor) -> torch.Tensor:
        x = self.norm1(inp)
        x = self.conv3(self.sg(self.conv2(self.conv1(x))))
        y = inp + x * self.beta
        x = self.norm2(y)
        x = self.conv5(self.sg(self.conv4(x)))
        return y + x * self.gamma

class WindowedDyTNAFRSSR(nn.Module):
    def __init__(self, in_channels: int = 3, width: int = 48, num_blks: int = 4, up_scale: int = 4, min_disp: int = -2, max_disp: int = 14, fixup_zero_init: bool = True):
        super().__init__()
        self.up_scale = up_scale
        self.intro = nn.Conv2d(in_channels, width, kernel_size=3, padding=1)
        self.stereo_fusion = WindowedSCAM(width, min_disp=min_disp, max_disp=max_disp)
        self.blocks = nn.ModuleList([DyTNAFBlock(width) for _ in range(num_blks)])
        self.up = nn.Sequential(
            nn.Conv2d(width, in_channels * (up_scale ** 2), kernel_size=3, padding=1),
            nn.PixelShuffle(up_scale)
        )
        if fixup_zero_init:
            nn.init.zeros_(self.up[0].weight)
            nn.init.zeros_(self.up[0].bias)
    def forward(self, img_l: torch.Tensor, img_r: torch.Tensor):
        base_l = F.interpolate(img_l, scale_factor=self.up_scale, mode='bicubic', align_corners=False)
        base_r = F.interpolate(img_r, scale_factor=self.up_scale, mode='bicubic', align_corners=False)
        fl = self.intro(img_l)
        fr = self.intro(img_r)
        fl, fr = self.stereo_fusion(fl, fr)
        for b in self.blocks:
            fl = b(fl)
            fr = b(fr)
        out_l = self.up(fl) + base_l
        out_r = self.up(fr) + base_r
        return torch.clamp(out_l, 0.0, 1.0), torch.clamp(out_r, 0.0, 1.0)

print(f'[*] Student Model WindowedDyTNAFRSSR defined! Total Params: {sum(p.numel() for p in WindowedDyTNAFRSSR().parameters()):,}')


## 5. Dual-T4 Training Loop with Multi-Dataset & High-Frequency Edge Loss

In [ ]:
class SobelLoss(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)
    def forward(self, pred, target):
        p_gray = 0.299 * pred[:, 0:1] + 0.587 * pred[:, 1:2] + 0.114 * pred[:, 2:3]
        t_gray = 0.299 * target[:, 0:1] + 0.587 * target[:, 1:2] + 0.114 * target[:, 2:3]
        gx_p = F.conv2d(p_gray, self.kx, padding=1)
        gy_p = F.conv2d(p_gray, self.ky, padding=1)
        gx_t = F.conv2d(t_gray, self.kx, padding=1)
        gy_t = F.conv2d(t_gray, self.ky, padding=1)
        return F.l1_loss(gx_p, gx_t) + F.l1_loss(gy_p, gy_t)

train_dataset = MultiCanonicalStereoDataset(crop_lr=48, upscale=4, is_train=True, max_pairs=1200)
val_dataset = MultiCanonicalStereoDataset(crop_lr=48, upscale=4, is_train=False, max_pairs=50)
batch_size = 16 * max(1, num_gpus)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=1)

base_model = WindowedDyTNAFRSSR(width=48, num_blks=4, up_scale=4, min_disp=-2, max_disp=14, fixup_zero_init=True).to(device)
if num_gpus > 1:
    model = nn.DataParallel(base_model)
else:
    model = base_model

sobel_loss_fn = SobelLoss().to(device)
epochs = 75
optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

best_psnr = 0.0
best_pth_path = '/kaggle/working/dyt_nafrssr_quest3s_best.pth'

print(f'[*] Starting Multi-Dataset Training on Dual Tesla T4 ({epochs} Epochs)...')
for epoch in range(1, epochs + 1):
    model.train()
    t0 = time.time()
    total_loss, total_pix, total_edge = 0.0, 0.0, 0.0
    steps = 0
    for batch in train_loader:
        lr_l = batch['lr_l'].to(device, non_blocking=True)
        lr_r = batch['lr_r'].to(device, non_blocking=True)
        hr_l = batch['hr_l'].to(device, non_blocking=True)
        hr_r = batch['hr_r'].to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            pred_l, pred_r = model(lr_l, lr_r)
            loss_pix = F.l1_loss(pred_l, hr_l) + F.l1_loss(pred_r, hr_r)
            loss_edge = (sobel_loss_fn(pred_l, hr_l) + sobel_loss_fn(pred_r, hr_r)) * 0.1
            loss = loss_pix + loss_edge
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        total_pix += loss_pix.item()
        total_edge += loss_edge.item()
        steps += 1
    scheduler.step()
    dt = time.time() - t0

    if epoch % 5 == 0 or epoch == epochs:
        model.eval()
        psnrs = []
        with torch.no_grad():
            for vbatch in val_loader:
                v_ll = vbatch['lr_l'].to(device)
                v_lr = vbatch['lr_r'].to(device)
                v_hl = vbatch['hr_l'].to(device)
                p_l, _ = model(v_ll, v_lr)
                mse = F.mse_loss(p_l, v_hl).item()
                if mse > 1e-10:
                    psnrs.append(10.0 * math.log10(1.0 / mse))
        val_psnr = float(np.mean(psnrs)) if psnrs else 0.0
        print(f'Epoch [{epoch:02d}/{epochs:02d}] ({dt:.2f}s) | Loss: {total_loss/max(1,steps):.4f} (Pix: {total_pix/max(1,steps):.4f}, Edge: {total_edge/max(1,steps):.4f}) | Val PSNR: {val_psnr:.2f} dB')
        if val_psnr > best_psnr:
            best_psnr = val_psnr
            raw_sd = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            torch.save(raw_sd, best_pth_path)
            print(f'    [+] Saved NEW BEST Checkpoint: {val_psnr:.2f} dB -> {best_pth_path}')
    else:
        print(f'Epoch [{epoch:02d}/{epochs:02d}] ({dt:.2f}s) | Loss: {total_loss/max(1,steps):.4f}')

print(f'[*] Training Completed! Best Val PSNR: {best_psnr:.2f} dB')


## 6. Export Trained Model to ONNX FP32 & FP16 for Quest 3S Sentis

In [ ]:
eval_model = WindowedDyTNAFRSSR(width=48, num_blks=4, up_scale=4, min_disp=-2, max_disp=14).to(device)
if os.path.exists(best_pth_path):
    eval_model.load_state_dict(torch.load(best_pth_path, map_location=device))
eval_model.eval()

dummy_l = torch.randn(1, 3, 48, 48, device=device)
dummy_r = torch.randn(1, 3, 48, 48, device=device)
onnx_fp32_path = '/kaggle/working/dyt_nafrssr_quest3s_fp32.onnx'

torch.onnx.export(
    eval_model,
    (dummy_l, dummy_r),
    onnx_fp32_path,
    input_names=['input_left', 'input_right'],
    output_names=['output_left', 'output_right'],
    opset_version=17,
    dynamic_axes={
        'input_left': {2: 'height', 3: 'width'},
        'input_right': {2: 'height', 3: 'width'},
        'output_left': {2: 'out_height', 3: 'out_width'},
        'output_right': {2: 'out_height', 3: 'out_width'}
    }
)
print(f'[*] Successfully exported ONNX FP32: {onnx_fp32_path} ({os.path.getsize(onnx_fp32_path)/1024:.1f} KB)')
print('[*] Training & Deployment Artifact Pipeline COMPLETE!')
